In [112]:
# 필요한 라이브러리 가져오기
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

In [113]:
# 데이터 가져오기
imdb = keras.datasets.imdb
(x_train, y_train),(x_test, y_test) = imdb.load_data(num_words=10000)

In [114]:
# 영화 리뷰는 x_train(텍스트 형태)에, 감성 정보는 y_train(0 또는 1)에 저장되어 있다.
len(x_train), len(x_test)

(25000, 25000)

In [115]:
# 학습 데이터 1개 뽑기
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [116]:
# 학습 데이터 2개 비교
len(x_train[0]), len(x_train[1])

(218, 189)

In [117]:
# 출력 데이터 확인
y_train[0], y_train[1]

(np.int64(1), np.int64(0))

In [118]:
# 각 label 별 개수 확인 (y의 data가 balanced 한지 확인하기)
np.unique(y_train, return_counts=True)

(array([0, 1]), array([12500, 12500]))

In [119]:
# 리뷰를 복원을 위한 정수 인덱스 딕셔너리 설정

# 단어 -> 정수 인덱스 딕셔너리
word_to_index = imdb.get_word_index()

# 처음 몇 개의 인덱스는 특수 용도를 사용된다.
word_to_index = {k:(v+3) for k,v in word_to_index.items()}
word_to_index["<PAD>"] = 0 # 문장을 채우는 기호(Padding)
word_to_index["<START>"] = 1 # 시작을 표시(Starting)
word_to_index["<UNK>"] = 2 # 알려지지 않은 토큰(unknown tokens)
word_to_index["<UNUSED>"] = 3

# 인덱스 4부터가 실제로 리뷰에 등장한 단어이다.

index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

In [120]:
# 리뷰 복원
print(' '.join([index_to_word[index] for index in x_train[0]]))

# 읽어보면 긍정 리뷰인 것을 알 수 있다.

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha

In [121]:
# 전처리 (Embedding 방식!!)

# 필요한 라이브러리를 가져오기
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import *
from tensorflow.keras.layers import Input

In [122]:
# 영화 리뷰의 크기도 각각 다르기 때문에 이것을 일정 크기 이하로 제한해야 함. 이때 사용하는 함수가 pad_seqeunce()
x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

len(x_train[0]), len(x_train[1])

(100, 100)

In [123]:
print(x_train[0])

[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


In [124]:
# 신경망 모델 구축
vocab_size = 10000

model = Sequential()
model.add(Input(shape=(100,)))
model.add(Embedding(vocab_size, 64))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │       409,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,049,729 (4.00 MB)

 Trainable params: 1,049,729 (4.00 MB)

 Non-trainable params: 0 (0.00 B)

In [125]:
# 모델 학습
history = model.fit(x_train, y_train, batch_size=64, epochs=20, verbose=1, validation_data=(x_test, y_test))

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.7584 - loss: 0.4684 - val_accuracy: 0.8455 - val_loss: 0.3466
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.9402 - loss: 0.1664 - val_accuracy: 0.8208 - val_loss: 0.4363
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9926 - loss: 0.0305 - val_accuracy: 0.8288 - val_loss: 0.5384
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.9990 - loss: 0.0055 - val_accuracy: 0.8340 - val_loss: 0.6044
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9997 - loss: 0.0026 - val_accuracy: 0.8321 - val_loss: 0.6614
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.8350 - val_loss: 0.7013
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 1.0000 - loss: 6.1188e-04 - val_accuracy: 0.8349 - val_loss: 0.7292
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 1.0000 - loss: 3.7210e-04 

In [126]:
# 모델 평가
results = model.evaluate(x_test, y_test, verbose=2)
print(results)

782/782 - 2s - 3ms/step - accuracy: 0.8184 - loss: 1.0601
[1.0600770711898804, 0.8184000253677368]


In [127]:
# 직접 작성한 리뷰(더미 데이터)로 테스트하기

review = "This movie was absolute garbage. The acting was terrible, the plot made no sense, and I fell asleep halfway through. It is the worst film I have seen this year. I would not recommend this to anyone. What a complete waste of time and money."

# 리뷰가 매우 부정적이다!!

In [128]:
import re

# 알페벳과 공백만 남기고 특수 문자들은 전부 삭제한 후 공백으로 바꾸는 정규식
review = re.sub("[^0-9a-zA-Z]+", " ", review).lower().strip()

In [129]:
# 단어를 하나씩 꺼내서 정수 인덱스로 변환
review_encoding = []

# 리뷰의 각 단어 대하여 반복
for w in review.split():
  index = word_to_index.get(w, 2) # 딕셔너리에 없으면 2 반환
  if index <= 10000:               # 단어의 개수는 10000이하
    review_encoding.append(index)
  else:
    review_encoding.append(word_to_index["<UNK>"])

# 2차원 리스트로 전달
test_input = pad_sequences([review_encoding], maxlen = 100)
value = model.predict(test_input) # 예측
if (value > 0.5):
  print("긍정적인 리뷰입니다.")
else:
  print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
긍정적인 리뷰입니다.
